### Импорты

In [1]:
import re
import uuid

In [2]:
path = "../data/docs"

with open(f"{path}/sp60_extracted_v3.md", "r", encoding="utf-8") as f:
    md_text = f.read()

### 1. Удаление оглавления

In [3]:
def remove_toc(text: str) -> str:
    """Удаляет оглавление между '## Содержание' и '## Введение'.
    Оглавление частично таблица, частично список — для RAG оно не нужно."""
    return re.sub(
        r'## Содержание\s*\n.*?## Введение',
        '## Содержание\n\n## Введение',
        text,
        flags=re.DOTALL,
    )

### 2. Формулы: $$$$ \u2192 $$ и защита блоков

In [4]:
def fix_formula_delimiters(text: str) -> str:
    """Заменяет $$$$ на $$ для совместимости с рендерерами и LLM."""
    return text.replace('$$$$', '$$')


def protect_blocks(text: str) -> tuple:
    """Заменяет формулы ($$...$$) и изображения (![...](...))\u00a0на плейсхолдеры,
    чтобы последующая обработка их не ломала. Возвращает (text, store)."""
    store = {}

    def _replace(match: re.Match) -> str:
        key = f"\u00a7PROTECTED_{uuid.uuid4().hex[:8]}\u00a7"
        store[key] = match.group(0)
        return key

    # Формулы: $$ ... $$ (одиночная строка или многострочные)
    text = re.sub(r'\$\$.*?\$\$', _replace, text, flags=re.DOTALL)

    # Изображения
    text = re.sub(r'!\[[^\]]*\]\([^)]+\)', _replace, text)

    return text, store


def restore_blocks(text: str, store: dict) -> str:
    """Восстанавливает формулы и изображения из плейсхолдеров."""
    for key, value in store.items():
        text = text.replace(key, value)
    return text

### 3. Склеивание разорванных строк

In [5]:
def join_broken_lines(text: str) -> str:
    """Заменяет одиночный перенос строки на пробел внутри абзаца,
    но не трогает таблицы и нумерованные определения."""
    blocks = re.split(r'\n\n', text)
    new_blocks = []
    for block in blocks:
        lines = block.split('\n')
        # Проверяем, похоже ли на таблицу (множество строк с |)
        pipe_lines = sum(1 for l in lines if l.strip().startswith('|'))
        if pipe_lines > len(lines) * 0.5 and pipe_lines >= 2:
            new_blocks.append(block)
            continue
        # Проверяем, содержит ли блок нумерованные определения (3.1, 3.1.1, 6.3.1...)
        # Если каждая строка начинается с числа или '-', не склеиваем
        numbered_lines = sum(
            1 for l in lines
            if re.match(r'^\s*-?\s*\d+\.\d+', l) or re.match(r'^\s*-?\s*\d+\.\d+\.\d+', l)
        )
        if numbered_lines > len(lines) * 0.5 and numbered_lines >= 2:
            new_blocks.append(block)
            continue
        # Заменяем одиночные переносы на пробел
        block = re.sub(r'(?<!\n)\n(?!\n)', ' ', block)
        new_blocks.append(block)
    return '\n\n'.join(new_blocks)

### 4. Объединение разорванных таблиц

In [6]:
def merge_split_tables(text: str) -> str:
    """Объединяет таблицы, разорванные на страницы.
    Удаляет маркеры 'Окончание таблицы'/'Продолжение таблицы' между частями."""
    lines = text.splitlines()
    new_lines = []
    i = 0
    while i < len(lines):
        line = lines[i]
        # Ищем начало таблицы: строка с '|' и следующая содержит '---'
        if line.strip().startswith('|') and i + 1 < len(lines) and '---' in lines[i + 1]:
            table = [line]
            i += 1
            table.append(lines[i])  # строка с ---
            i += 1
            # Строки данных
            while i < len(lines) and lines[i].strip().startswith('|'):
                table.append(lines[i])
                i += 1
            # Пропускаем пустые строки после таблицы
            while i < len(lines) and lines[i].strip() == '':
                i += 1
            # Пропускаем маркер продолжения/окончания
            while i < len(lines) and re.match(
                r'^\s*(Окончание|Продолжение)\s+таблицы', lines[i].strip()
            ):
                i += 1
                while i < len(lines) and lines[i].strip() == '':
                    i += 1
            # Продолжение таблицы (строки с '|', но без '---' в первой строке)
            while i < len(lines) and lines[i].strip().startswith('|'):
                if '---' in lines[i]:
                    break
                table.append(lines[i])
                i += 1
            new_lines.extend(table)
        else:
            new_lines.append(line)
            i += 1
    return '\n'.join(new_lines)

### 5. Исправление ложных заголовков

In [7]:
# Ключевые слова, которые указывают, что текст после номера пункта — тело, а не название
_FALSE_HEADER_VERBS = re.compile(
    r'(?<!\w)('
    r'следует|должн|допускает|необходимо|рекомендует|'
    r'принимать|принимают|надлежит|требует|'
    r'запрещает|не допуска|подлежит|предусмотр'
    r')',
    re.IGNORECASE,
)


def _looks_like_sentence_continuation(line: str) -> bool:
    """Проверяет, выглядит ли строка как продолжление предложения."""
    stripped = line.strip()
    if not stripped:
        return False
    return stripped[0].islower()


def fix_false_headers(text: str) -> str:
    """Исправляет ложные заголовки.

    Два типа проблем:
    1. Заголовок содержит тело пункта (длинный текст, глаголы).
       Если в заголовке после '## N.M' есть ещё один номер пункта —
       отделяем заголовок от тела.
    2. Заголовок обрывается и продолжается на следующей строке —
       склеиваем и снимаем ##.
    """
    lines = text.split('\n')
    result = []
    i = 0
    while i < len(lines):
        line = lines[i]
        m = re.match(r'^(#{1,3})\s+(\d+(?:\.\d+){0,2})\s+(.+)$', line)
        if m:
            hashes, num, rest = m.group(1), m.group(2), m.group(3).rstrip()

            # Проверяем, есть ли внутри rest ещё один номер пункта
            # (например: "Системы отопления 6.2.1 В проектной...")
            inner_num = re.search(
                r'^(.+?)\s+(\d+\.\d+(?:\.\d+)?\s+\S.*)$', rest, re.DOTALL
            )

            if inner_num:
                title_part = inner_num.group(1).strip()
                body_part = inner_num.group(2).strip()
                # title_part — настоящее название раздела, body_part — тело пункта
                # Оставляем заголовок, тело переносим на следующую строку
                result.append(f'{hashes} {num} {title_part}')
                result.append(body_part)
                i += 1
                continue

            is_false = len(rest) > 80 or _FALSE_HEADER_VERBS.search(rest)

            if not is_false:
                j = i + 1
                while j < len(lines) and lines[j].strip() == '':
                    j += 1
                if j < len(lines) and not lines[j].startswith('#'):
                    next_line = lines[j].strip()
                    if re.search(r'[-–,]\s*$', rest) or _looks_like_sentence_continuation(next_line):
                        rest = rest + ' ' + next_line
                        i = j
                        is_false = True

            if is_false:
                result.append(rest)
            else:
                result.append(line)
        else:
            result.append(line)
        i += 1
    return '\n'.join(result)

### 5b. Иерархия заголовков

In [8]:
def fix_header_levels(text: str) -> str:
    """Выстраивает иерархию заголовков по номерам разделов.

    Правила:
    - '## N название' (одно число) — раздел, остаётся ##
    - '## N.M название' (два числа) — подраздел, понижается до ###
    - '## N.M.K название' (три числа) — редко, понижается до ####
    """
    lines = text.split('\n')
    result = []
    for line in lines:
        m = re.match(r'^(#{1,3})\s+(\d+)(?:\.(\d+)(?:\.(\d+))?)?\s+(.+)$', line)
        if m:
            hashes = m.group(1)
            major = m.group(2)
            minor = m.group(3)
            sub = m.group(4)
            title = m.group(5)

            if minor and not sub:
                # N.M — подраздел → ###
                result.append(f'### {major}.{minor} {title}')
            elif sub:
                # N.M.K — под-подраздел → ####
                result.append(f'#### {major}.{minor}.{sub} {title}')
            else:
                # N — раздел, оставляем ##
                result.append(f'## {major} {title}')
        else:
            result.append(line)
    return '\n'.join(result)

### 6. Исправление списков

In [9]:
def fix_lists(text: str) -> str:
    """Исправляет маркировку списков и нумерованных пунктов.

    - Убирает '-' перед нумерованными пунктами ('- 3.1 ...' → '3.1 ...')
    - Убирает '-' перед именованными перечислениями ('- д) ...' → 'д) ...')
    - Разделяет склеенные пункты на отдельные абзацы
    - Исправляет '-слово' → '- слово', '- -слово' → '- слово'
    """
    # --- 1. Разделяем склеенные '- 3.1.xxx ...' в середине строк ---
    # Но не трогаем строки-заголовки (начинаются с #)
    text = re.sub(r'(?<!^)(?<!\n)(?<!#)\s+-\s+(\d+\.\d+(?:\.\d+)?\s+\S)', r'\n\1', text)
    # '- X.Y слово' в середине строки (без дефиса) тоже разделяем
    # Но не трогаем заголовки — только после двоеточия, точки с запятой, точки
    text = re.sub(r'(?<=[;:.])\s+(\d+\.\d+\s+\S)', r'\n\1', text)

    # --- 2. Убираем '-' перед нумерованными пунктами в начале строки ---
    text = re.sub(r'(?m)^-\s+(\d+\.\d+(?:\.\d+)?\s+)', r'\1', text)

    # --- 3. Разделяем именованные перечисления '- а) ... - б) ...' ---
    text = re.sub(r'\s+-\s+([а-яё]\))\s*', r'\n\1 ', text)

    # --- 4. Убираем '-' перед именованными перечислениями в начале строки ---
    text = re.sub(r'(?m)^-\s+([а-яё]\))\s*', r'\1 ', text)

    # --- 5. Двойной дефис: '- -слово' → '- слово' ---
    text = re.sub(r'(?m)^-\s+-(\S)', r'- \1', text)

    # --- 6. Нет пробела после дефиса: '-слово' → '- слово' ---
    text = re.sub(r'(?m)^-(\S)', r'- \1', text)

    # --- 7. В середине строки: ': -слово' → ': \n- слово' ---
    text = re.sub(r'(?<=[;:])\s+-(\S)', r'\n- \1', text)

    # --- 8. Отделяем список от предыдущего абзаца ---
    lines = text.split('\n')
    result = []
    for i, line in enumerate(lines):
        stripped = line.strip()
        if stripped.startswith('-') or re.match(r'^[а-яё]\)', stripped):
            if i > 0 and lines[i - 1].strip() != '' and not lines[i - 1].strip().startswith('-'):
                result.append('')
        result.append(line)
    return '\n'.join(result)

### 7. Примечания

In [10]:
def format_notes(text: str) -> str:
    """Заменяет заголовки '## Примечания' на жирный текст."""
    text = re.sub(r'(?m)^#{1,3}\s+(Примечани[ея])(.*)$', r'**\1\2**', text)
    text = re.sub(r'\*\*\s+', '** ', text)
    return text


def separate_notes(text: str) -> str:
    """Вставляет пустую строку перед **Примечания**, если её нет."""
    text = re.sub(r'(?<!\n\n)(\*\*Примечани[ея]\*\*)', r'\n\n\1', text)
    return text


def fix_note_enumerations(text: str) -> str:
    """Форматирует нумерованные списки внутри блока **Примечания**."""
    paragraphs = re.split(r'\n\n', text)
    for i, para in enumerate(paragraphs):
        if para.startswith('**Примечания**') or para.startswith('**Примечание**'):
            para = re.sub(r' +', ' ', para)
            para = re.sub(r'(\*\*Примечани[ея]\*\*)\s+', r'\1\n', para)
            para = re.sub(r'(?<=\n)(\d+)\s+', r'\1. ', para)
            para = re.sub(r'(\*\*Примечани[ея]\*\*)\n?(\d+)\s+', r'\1\n\2. ', para)
            paragraphs[i] = para
    return '\n\n'.join(paragraphs)

### 8. Очистка лишних пустых строк и артефактов

In [11]:
def clean_blank_lines(text: str) -> str:
    """Финальная очистка от лишних пустых строк."""
    return re.sub(r'\n{3,}', '\n\n', text)

### Пайплайн постпроцессинга

In [12]:
def postprocess_md(md_text: str) -> str:
    """Полный постпроцессинг."""
    # 1. Удаление оглавления
    md_text = remove_toc(md_text)

    # 2. Формулы: $$$$ → $$
    md_text = fix_formula_delimiters(md_text)

    # 3. Защита формул и изображений плейсхолдерами
    md_text, store = protect_blocks(md_text)

    # 4. Склеивание разорванных строк (формулы и картинки уже защищены)
    md_text = join_broken_lines(md_text)

    # 5. Объединение разорванных таблиц
    md_text = merge_split_tables(md_text)

    # 6. Исправление ложных заголовков
    md_text = fix_false_headers(md_text)

    # 7. Иерархия заголовков (## 6.1 → ### 6.1)
    md_text = fix_header_levels(md_text)

    # 8. Исправление списков и нумерованных пунктов
    md_text = fix_lists(md_text)

    # 9. Примечания
    md_text = format_notes(md_text)
    md_text = separate_notes(md_text)
    md_text = fix_note_enumerations(md_text)

    # 10. Очистка лишних пустых строк
    md_text = clean_blank_lines(md_text)

    # 11. Восстановление формул и изображений
    md_text = restore_blocks(md_text, store)

    return md_text

### Запуск

In [13]:
md_clean = postprocess_md(md_text)

In [14]:
with open(f"{path}/sp60_final.md", "w", encoding="utf-8") as f:
    f.write(md_clean)